In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image
from typing import TypedDict


In [ ]:
# define state
class BatsmanState(TypedDict):
    name: str
    runs: int
    balls: int
    fours: int
    sixes: int
    sr: float
    bpb: float
    boundary_percent: float
    summary: str


In [ ]:
def calc_sr(state: BatsmanState) -> dict:
    balls = state['balls']
    sr = (state['runs'] / balls) * 100 if balls > 0 else 0.0
    return {'sr': round(sr, 2)}


In [ ]:
def calc_bpb(state: BatsmanState) -> dict:
    boundaries = state['fours'] + state['sixes']
    bpb = state['balls'] / boundaries if boundaries > 0 else float('inf')
    return {'bpb': round(bpb, 2) if bpb < float('inf') else bpb}


In [ ]:
def calc_boundary_percent(state: BatsmanState) -> dict:
    runs = state['runs']
    boundary_bonus = state['fours'] * 4 + state['sixes'] * 6
    boundary_percent = (boundary_bonus / runs) * 100 if runs > 0 else 0.0
    return {'boundary_percent': round(boundary_percent, 2)}


In [ ]:
def calc_summary(state: BatsmanState) -> dict:
    summary = f"""
{state['name']} scored {state['runs']} runs off {state['balls']} balls.

Strike Rate           - {state['sr']}
Balls per boundary    - {state['bpb']}
Boundary Percent      - {state['boundary_percent']}"""
    return {'summary': summary}


In [ ]:
graph = StateGraph(BatsmanState)

graph.add_node('calc_sr', calc_sr)
graph.add_node('calc_bpb', calc_bpb)
graph.add_node('calc_boundary_percent', calc_boundary_percent)
graph.add_node('calc_summary', calc_summary)

# parallel fan-out, fan-in to summary
graph.add_edge(START, 'calc_sr')
graph.add_edge(START, 'calc_bpb')
graph.add_edge(START, 'calc_boundary_percent')
graph.add_edge('calc_sr', 'calc_summary')
graph.add_edge('calc_bpb', 'calc_summary')
graph.add_edge('calc_boundary_percent', 'calc_summary')
graph.add_edge('calc_summary', END)

workflow = graph.compile()


In [ ]:
initial_state = {
    'name': 'Virat Kohli',
    'runs': 100,
    'balls': 40,
    'fours': 5,
    'sixes': 8,
}

result = workflow.invoke(initial_state)
print(result['summary'])


In [ ]:
Image(workflow.get_graph().draw_mermaid_png())
